In [22]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np

def parse_xml_to_df(xml_path, entity_tag='book'):
    """Parses XML files into a DataFrame handling scalar and list elements."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    records = []
    
    for entity in root.findall(f'.//{entity_tag}'):
        data = {}
        for child in entity:
            if len(child) > 0:
                vals = [sub.text for sub in child if sub.text]
                data[child.tag] = vals if vals else None
            else:
                data[child.tag] = child.text if child.text else None
        records.append(data)
    return pd.DataFrame(records)

def clean_missing_values(df):
    """Replaces empty strings, empty lists, and 'None' strings with np.nan safely."""
    def is_empty(x):
        if x is None:
            return True
        if isinstance(x, (list, np.ndarray, tuple)):
            return len(x) == 0
        if pd.isna(x):
            return True
        if isinstance(x, str):
            return x.strip() in ["", "[]", "None", "nan", "NaN"]
        return False

    return df.map(lambda x: np.nan if is_empty(x) else x)

In [23]:
# 1. Load DBpedia XML
df_dbpedia = parse_xml_to_df('data/dbpedia_books.xml', entity_tag='book')

# 2. Load Goodreads CSV
df_csv = pd.read_csv('data/books_1.Best_Books_Ever.csv')

# Datasets Dictionary for automated processing
datasets = {
    'DBpedia (XML)': df_dbpedia,
    'Goodreads (CSV)': df_csv
    # 'Dataset 3': df_third  <-- Here you can add the 3rd required dataset
}

In [24]:
# ==========================================
# MISSING VALUES ANALYSIS (>30% THRESHOLD)
# ==========================================
missing_reports = []

for name, df in datasets.items():
    df_cleaned = clean_missing_values(df)
    missing_pct = (df_cleaned.isnull().mean() * 100).round(2)
    missing_counts = df_cleaned.isnull().sum()
    total_records = len(df_cleaned)
    
    for col in df_cleaned.columns:
        pct = missing_pct[col]
        missing_reports.append({
            'Dataset': name,
            'Attribute Name': col,
            'Missing Count': missing_counts[col],
            'Total Records': total_records,
            'Missing (%)': pct,
            'High Missing (>30%)': pct > 30.0
        })

df_missing_summary = pd.DataFrame(missing_reports)

# Display attributes with >30% missing values
high_missing_df = df_missing_summary[df_missing_summary['High Missing (>30%)']].sort_values(by=['Dataset', 'Missing (%)'], ascending=[True, False])

print("=== ATTRIBUTES WITH > 30% MISSING VALUES ===")
if not high_missing_df.empty:
    display(high_missing_df[['Dataset', 'Attribute Name', 'Missing Count', 'Total Records', 'Missing (%)']])
else:
    print("No attributes found with more than 30% missing values.")

=== ATTRIBUTES WITH > 30% MISSING VALUES ===


,Dataset,Attribute Name,Missing Count,Total Records,Missing (%)
5,DBpedia (XML),avg_star_rating,19434,19434,100.00
6,DBpedia (XML),number_of_reviews,19434,19434,100.00
2,DBpedia (XML),year,17189,19434,88.45
4,DBpedia (XML),language,16905,19434,86.99
9,DBpedia (XML),genres,9386,19434,48.30
20,Goodreads (CSV),edition,47523,52478,90.56
25,Goodreads (CSV),awards,41864,52478,79.77
29,Goodreads (CSV),setting,40900,52478,77.94
18,Goodreads (CSV),characters,38712,52478,73.77
11,Goodreads (CSV),series,29009,52478,55.28


In [25]:
# ==========================================
# TABLE 1: DATASET PROFILE & ATTRIBUTE LIST WITH (MV)
# ==========================================
table_1_rows = []

for name, df in datasets.items():
    df_cleaned = clean_missing_values(df)
    missing_pct = df_cleaned.isnull().mean() * 100
    
    # Format attribute list and mark >30% missing values with (MV)
    formatted_attrs = []
    for col in df_cleaned.columns:
        if missing_pct[col] > 30:
            formatted_attrs.append(f"{col} (MV)")
        else:
            formatted_attrs.append(col)
            
    table_1_rows.append({
        'Dataset': name,
        'Source(*)': 'data/dbpedia_books.xml' if 'XML' in name else 'data/books_1.Best_Books_Ever.csv',
        'Format': 'xml' if 'XML' in name else 'csv',
        '# of entities': len(df_cleaned),
        '# of attributes': len(df_cleaned.columns),
        'List of attributes (**)': ", ".join(formatted_attrs)
    })

df_table_1 = pd.DataFrame(table_1_rows)
print("=== TABLE 1: SCHEMA AND BASIC PROFILE OF EACH DATASET ===")
display(df_table_1)

=== TABLE 1: SCHEMA AND BASIC PROFILE OF EACH DATASET ===


,Dataset,Source(*),Format,# of entities,# of attributes,List of attributes (**)
0,DBpedia (XML),data/dbpedia_books.xml,xml,19434,10,"title, publisher, year (MV), pages, language (..."
1,Goodreads (CSV),data/books_1.Best_Books_Ever.csv,csv,52478,23,"title, series (MV), author, rating, descriptio..."


In [30]:
import pandas as pd

# ==========================================
# 1. DYNAMIC ATTRIBUTE TYPE INFERENCE
# ==========================================
def infer_attribute_type(series):
    """
    Infers semantic attribute type (string, string/list, numeric, date) 
    based on the actual data content of a series.
    """
    valid_vals = series.dropna()
    if len(valid_vals) == 0:
        return 'string'
    
    sample = valid_vals.iloc[0]
    
    # Check for List types
    if isinstance(sample, (list, tuple)):
        return 'string/list'
    if isinstance(sample, str) and sample.strip().startswith('['):
        return 'string/list'
    
    # Check for Numeric types
    if pd.api.types.is_numeric_dtype(series):
        return 'numeric'
    
    # Check for Date / Year types
    col_name = series.name.lower()
    if 'date' in col_name or 'year' in col_name:
        return 'date'
        
    return 'string'

# ==========================================
# 2. DYNAMIC TABLE 2 GENERATION
# ==========================================
all_columns = set()
for df in datasets.values():
    all_columns.update(df.columns)

table_2_rows = []

for col in sorted(list(all_columns)):
    # Identify which datasets contain this attribute
    found_in = [ds_name for ds_name, df in datasets.items() if col in df.columns]
    
    # Get a sample dataframe that contains this column to infer its type
    sample_df = next(df for df in datasets.values() if col in df.columns)
    attr_type = infer_attribute_type(sample_df[col])
    
    table_2_rows.append({
        'Attribute name': col,
        'Attribute type': attr_type,
        'Datasets in which found': ", ".join(found_in),
        'Count Datasets': len(found_in)
    })

df_table_2 = pd.DataFrame(table_2_rows)

# Display Full Table 2
print("=== TABLE 2: INTEGRATED SCHEMA OVERLAP (ALL ATTRIBUTES) ===")
display(df_table_2[['Attribute name', 'Attribute type', 'Datasets in which found']])

# ==========================================
# 3. OVERLAPPING ATTRIBUTES ONLY (COMPACT TABLE)
# ==========================================
# Filter attributes that appear in 2 or more datasets
df_overlapping = df_table_2[df_table_2['Count Datasets'] >= 2].drop(columns=['Count Datasets'])

print("\n=== OVERLAPPING ATTRIBUTES ONLY (FOUND IN AT LEAST 2 DATASETS) ===")
if not df_overlapping.empty:
    display(df_overlapping)
else:
    print("No exact overlapping attribute names found between the datasets.")

# ==========================================
# 4. ENTITY OVERLAP CHECK (TITLE & TITLE + AUTHOR)
# ==========================================
clean_str = lambda s: str(s).lower().strip() if pd.notna(s) else ""

def get_first_author(val):
    if isinstance(val, list) and len(val) > 0:
        return clean_str(val[0])
    return clean_str(val)

# A) Title Exact Match
titles_dbpedia = set(df_dbpedia['title'].apply(clean_str)) - {""}
titles_csv = set(df_csv['title'].apply(clean_str)) - {""}
overlap_titles = titles_dbpedia.intersection(titles_csv)

# B) Title + First Author Match (Significantly more precise)
df_dbpedia['key'] = df_dbpedia['title'].apply(clean_str) + " | " + df_dbpedia['authors'].apply(get_first_author)
df_csv['key'] = df_csv['title'].apply(clean_str) + " | " + df_csv['author'].apply(get_first_author)

keys_dbpedia = set(df_dbpedia[df_dbpedia['title'].notnull()]['key']) - {" | "}
keys_csv = set(df_csv[df_csv['title'].notnull()]['key']) - {" | "}
overlap_keys = keys_dbpedia.intersection(keys_csv)

print("\n=== ENTITY OVERLAP CHECK ===")
print(f"1. Overlap by Title (Exact Match):    {len(overlap_titles)} books")
print(f"2. Overlap by Title + Author:         {len(overlap_keys)} books")

=== TABLE 2: INTEGRATED SCHEMA OVERLAP (ALL ATTRIBUTES) ===


,Attribute name,Attribute type,Datasets in which found
0,author,string,Goodreads (CSV)
1,authors,string/list,DBpedia (XML)
2,avg_star_rating,string,DBpedia (XML)
3,awards,string/list,Goodreads (CSV)
4,bbeScore,numeric,Goodreads (CSV)
5,bbeVotes,numeric,Goodreads (CSV)
6,bookFormat,string,Goodreads (CSV)
7,characters,string/list,Goodreads (CSV)
8,description,string,Goodreads (CSV)
9,edition,string,Goodreads (CSV)



=== OVERLAPPING ATTRIBUTES ONLY (FOUND IN AT LEAST 2 DATASETS) ===


,Attribute name,Attribute type,Datasets in which found
11,genres,string/list,"DBpedia (XML), Goodreads (CSV)"
14,key,string,"DBpedia (XML), Goodreads (CSV)"
15,language,string,"DBpedia (XML), Goodreads (CSV)"
19,pages,string,"DBpedia (XML), Goodreads (CSV)"
22,publisher,string,"DBpedia (XML), Goodreads (CSV)"
27,title,string,"DBpedia (XML), Goodreads (CSV)"



=== ENTITY OVERLAP CHECK ===
1. Overlap by Title (Exact Match):    2583 books
2. Overlap by Title + Author:         1075 books
